# Tugas Kelompok 2: Data Collection Pipeline

## Kelompok sowang

1. Fransisca Ellya Bunaren (2306152286)
2. Micheline Wijaya Limbergh (2306207013)
3. Nisrina Annaisha Sarnadi (2306275960)
4. Rizqya Az Zahra Putri (2306244936)
5. Muhammad Satrio Haryo Kusumo (2006597771)

# Preprocessing untuk Data Analisis Sentimen (RQ2)

Tahap ini digunakan untuk menyiapkan teks dan menghasilkan prediksi sentimen untuk setiap postingan. Fokusnya adalah menjaga informasi yang penting bagi sentimen, seperti kata negasi, emoji, dan tanda baca ekspresif. Di bagian akhir, data RQ2 dicocokkan dengan data hasil preprocessing RQ1 berdasarkan ID postingan.

## 0. Konfigurasi file



In [1]:
import re
import pandas as pd
import torch
import html

from pathlib import Path

ROOT = Path.cwd().parent

INPUT = ROOT / "data" / "labeling" / "relevant_final.csv"
OUT = ROOT / "data" / "processed" / "rq2"

OUT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"
BATCH_SIZE = 16
MAX_LEN = 512

print("Input:", INPUT)
print("Output:", OUT)
print("GPU tersedia:", torch.cuda.is_available())

Input: c:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang\data\labeling\relevant_final.csv
Output: c:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang\data\processed\rq2
GPU tersedia: False


## 1. Membaca data yang sudah lolos seleksi

Data diambil dari `relevant_final.csv`. Di sini dipastikan ID setiap postingan terisi dan tidak ada duplikat agar hasil sentimen nantinya bisa disambungkan kembali ke data lain.

In [2]:
df = pd.read_csv(INPUT, dtype={"id": str})

print("Jumlah postingan:", len(df))
print("Kolom:", df.columns.tolist())

display(df[["id", "text"]].head())

Jumlah postingan: 8072
Kolom: ['month', 'group', 'keyword', 'id', 'created_at', 'lang', 'text', 'conversation_id', 'like_count', 'reply_count', 'repost_count', 'collected_at', 'source_month', 'word_count', 'core_word_count', 'is_short', 'is_empty_core', 'is_reply', 'is_dup_core', 'n_duplicates_removed', 'is_non_literal', 'strong_hits', 'strong_terms', 'weak_hits', 'weak_terms', 'filter_reason', 'filter_otomatis']


,id,text
0,1973219344600760446,"Lebih butuh konseling pra- nikah yg mendalam, ..."
1,1973223454599262597,@toe_giman Miris... Akibat dr pny ortu tp ga b...
2,1973230829380968860,@sopir_idiot Kalau kayak gini. Ngasitaunya har...
3,1973234406467969447,@kitoorah @tanyakanrl mau pemerintah bikin atu...
4,1973234962532045079,Nanti deh mau ngumpulin video2 xinlong yg kusu...


## 2. Membersihkan teks untuk analisis sentimen

- Penghapusan URL dan mention: URL dan mention dihapus dari teks yang digunakan untuk model.
- Perapian spasi: Spasi dirapikan agar format teks lebih konsisten.
- Mempertahankan konteks: Kata negasi (misalnya tidak atau nggak), slang, emoji, dan tanda baca tetap dipertahankan karena dapat membantu mengidentifikasi nada postingan.
- Penyimpanan teks asli: Teks asli tetap disimpan pada kolom `text` untuk keperluan pengecekan.

In [3]:
def preprocess_sentiment(text):
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"https?://\S+|www\.\S+|\b\w+\.(?:co|com|id)/\S+", " ", text)
    text = re.sub(r"(?<!\w)@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"(.)\1{3,}", r"\1\1", text)
    return re.sub(r"\s+", " ", text).strip()

df["text_sentiment"] = df["text"].apply(preprocess_sentiment)

print("Teks kosong:", df["text_sentiment"].eq("").sum())

display(df[["text", "text_sentiment"]].head(10))
jumlah_url = df["text_sentiment"].str.contains(r"https?://|www\.",regex=True,na=False).sum()

print("URL tersisa di text_sentiment:", jumlah_url)

Teks kosong: 0


,text,text_sentiment
0,"Lebih butuh konseling pra- nikah yg mendalam, ...","Lebih butuh konseling pra- nikah yg mendalam, ..."
1,@toe_giman Miris... Akibat dr pny ortu tp ga b...,Miris... Akibat dr pny ortu tp ga bs bikin tel...
2,@sopir_idiot Kalau kayak gini. Ngasitaunya har...,Kalau kayak gini. Ngasitaunya harus pakai gent...
3,@kitoorah @tanyakanrl mau pemerintah bikin atu...,mau pemerintah bikin aturan gmnpun kalo emng p...
4,Nanti deh mau ngumpulin video2 xinlong yg kusu...,Nanti deh mau ngumpulin video2 xinlong yg kusu...
5,@S0UYYA Keren parenting nya,Keren parenting nya
6,@tanyakanrl -- ada yg efeknya sampe gamau seko...,-- ada yg efeknya sampe gamau sekolah. Padahal...
7,Setelah lihat anak tantrum dan ortu hrs nuruti...,Setelah lihat anak tantrum dan ortu hrs nuruti...
8,Kapan lagi ini skzoo-denimalz mengobrol membah...,Kapan lagi ini skzoo-denimalz mengobrol membah...
9,"@meinmokhtar Masu kelas parenting, fikir nama ...","Masu kelas parenting, fikir nama anak, pampers..."


URL tersisa di text_sentiment: 0


In [4]:
# Validasi hasil preprocessing
assert "id" in df.columns
assert "text" in df.columns
assert df["id"].notna().all()
assert df["id"].is_unique

print("Jumlah data:", len(df))
print("Teks kosong:", df["text_sentiment"].eq("").sum())

# Simpan hasil preprocessing
PREPROCESSED_OUTPUT = OUT / "sentiment_preprocess_output.csv"

df.to_csv(
    PREPROCESSED_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Hasil preprocessing disimpan di:", PREPROCESSED_OUTPUT)

Jumlah data: 8072
Teks kosong: 0
Hasil preprocessing disimpan di: c:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang\data\processed\rq2\sentiment_preprocess_output.csv
